# Задание 3: Ансамблевая модель для Adult Income

В работе рассматривается задача бинарной классификации на датасете **Adult Income**: по социально-демографическим и экономическим характеристикам требуется предсказать, относится ли наблюдение к классу дохода `>50K` или `<=50K`. Постановка относится к задачам табличного машинного обучения с дисбалансом классов, поэтому ключевое значение имеют корректная схема валидации, отсутствие утечки данных и содержательная интерпретация результатов.

Используется тот же датасет, что и в `ДЗ_2`, что обеспечивает сопоставимость с ранее построенным baseline. В текущем этапе задача состоит не в замене исходных данных или протокола оценки, а в повышении качества за счёт более выразительной ансамблевой модели и более глубокого анализа её решений.


## 1. Цель эксперимента и воспроизводимость

Цель эксперимента состоит в том, чтобы получить модель, которая статистически обоснованно превосходит baseline по качеству классификации положительного класса `>50K`, при этом остаётся интерпретируемой на глобальном и локальном уровнях.

Для воспроизводимости фиксируются `RANDOM_STATE = 42` и `numpy` seed. Это обеспечивает повторяемость разбиения данных, процедуры кросс-валидации и итоговых метрик при запуске ноутбука сверху вниз.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings(
    "ignore",
    message="A worker stopped while some jobs were given to the executor.*",
)
warnings.filterwarnings(
    "ignore",
    message="The NumPy global RNG was seeded.*",
    category=FutureWarning,
)

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
    }
)

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "ДЗ_3").exists():
    FIGURES_DIR = PROJECT_ROOT / "ДЗ_3" / "figures"
else:
    FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 2. Данные и постановка задачи

Данные загружаются из Adult Income через `fetch_ucirepo(id=2)`. Целевая переменная — `income`, где положительный класс соответствует доходу `>50K`. Такая постановка непосредственно воспроизводит предыдущий этап и позволяет проводить прямое сравнение новой модели с baseline.

После загрузки выполняется унификация названий столбцов. Это технический, но принципиально важный шаг: единый формат имён уменьшает риск ошибок в последующей предобработке и делает структуру признакового пространства прозрачной.


In [ ]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

df = pd.concat([X, y], axis=1)

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

print(f"Dataset shape: {df.shape}")
display(df.head())


Dataset shape: (48842, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Предобработка данных

В выборке присутствуют признаки двух типов. К числовым относятся `age`, `fnlwgt`, `education_num`, `capital_gain`, `capital_loss`, `hours_per_week`; к категориальным — `workclass`, `education`, `marital_status`, `occupation`, `relationship`, `race`, `sex`, `native_country`. Для этих групп используются разные преобразования, что делает `ColumnTransformer` методически оправданным.

Для числовых признаков применяется `SimpleImputer(strategy="median")`. Медианная импутация устойчива к выбросам и подходит для распределений, в которых возможны асимметрия и редкие экстремальные значения. Для категориальных признаков используются `SimpleImputer(strategy="most_frequent")` и `OneHotEncoder(handle_unknown="ignore")`: первый восстанавливает пропуски без искусственного введения новых категорий, второй переводит категориальные признаки в форму, пригодную для обучения модели, и защищает пайплайн от ошибок на ранее не встречавшихся значениях.

Вся предобработка встроена в `Pipeline` и обучается только на train-части. Это принципиально исключает утечку информации из тестовой выборки и обеспечивает корректность последующей оценки качества.


In [ ]:
object_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

df[object_columns] = df[object_columns].apply(lambda column: column.str.strip())
df[object_columns] = df[object_columns].replace("?", np.nan)
df["income"] = df["income"].str.replace(".", "", regex=False)

missing_values = (
    df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame()
)
missing_values = missing_values[missing_values["missing_count"] > 0]

feature_df = df.drop(columns="income")
numeric_features = feature_df.select_dtypes(include="number").columns.tolist()
categorical_features = feature_df.select_dtypes(exclude="number").columns.tolist()

income_distribution = (
    df["income"].value_counts(dropna=False).rename_axis("income").to_frame("count")
)
income_distribution["share"] = (
    income_distribution["count"] / income_distribution["count"].sum()
).round(4)

print("Missing values:")
display(missing_values)
print("Income distribution:")
display(income_distribution)
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")


Missing values:


,missing_count
occupation,2809
workclass,2799
native_country,857


Income distribution:


,count,share
income,,
<=50K,37155,0.7607
>50K,11687,0.2393


Numeric features: 6
Categorical features: 8


## 4. Формирование целевой переменной и разделение данных

Целевая переменная кодируется как `<=50K -> 0`, `>50K -> 1`. Такое представление задаёт однозначную интерпретацию положительного класса и согласовано с последующей оценкой через `F1-score`.

Данные делятся на train/test в пропорции `80/20` с `stratify=y`. Стратификация сохраняет исходный баланс классов в обеих частях выборки и делает сравнение моделей корректным в условиях дисбаланса.


In [ ]:
target_mapping = {"<=50K": 0, ">50K": 1}

X = feature_df.copy()
y = df["income"].map(target_mapping)

if y.isna().any():
    unexpected_labels = sorted(df.loc[y.isna(), "income"].dropna().unique())
    raise ValueError(f"Unexpected target labels: {unexpected_labels}")

y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "rows": [X_train.shape[0], X_test.shape[0]],
        "positive_rate": [y_train.mean(), y_test.mean()],
    }
)

display(split_summary)


,dataset,rows,positive_rate
0,train,39073,0.239270
1,test,9769,0.239328


## 5. Baseline для сравнения

Вначале рассчитывается нижняя граница качества через `DummyClassifier`, затем воспроизводится baseline-модель `LogisticRegression`, использованная ранее. Такое сравнение необходимо, чтобы измерять не абстрактное значение метрики, а реальный прирост качества по отношению к простому ориентиру и к интерпретируемому базовому решению.

Наличие baseline также позволяет отделить содержательное улучшение модели от случайных колебаний результата, связанных с конкретным разбиением данных.


In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)
dummy_pred = dummy_model.predict(X_test)
dummy_f1 = f1_score(y_test, dummy_pred, zero_division=0)

baseline_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

baseline_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

baseline_preprocessor = ColumnTransformer(
    transformers=[
        ("num", baseline_numeric_transformer, numeric_features),
        ("cat", baseline_categorical_transformer, categorical_features),
    ]
)

baseline_logreg = Pipeline(
    steps=[
        ("preprocessor", baseline_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                solver="liblinear",
            ),
        ),
    ]
)

baseline_logreg.fit(X_train, y_train)
baseline_pred = baseline_logreg.predict(X_test)
baseline_logreg_f1 = f1_score(y_test, baseline_pred, zero_division=0)

baseline_table = pd.DataFrame(
    {
        "model": ["Dummy", "LogisticRegression (ДЗ_2 baseline)"],
        "f1_test": [dummy_f1, baseline_logreg_f1],
    }
)

display(baseline_table.round(4))


,model,f1_test
0,Dummy,0.0000
1,LogisticRegression (ДЗ_2 baseline),0.6558


## 6. Выбор модели

В качестве основной модели используется `RandomForestClassifier`. Это ансамбль деревьев решений, который принципиально сильнее линейного baseline в задачах, где важны нелинейные зависимости и взаимодействия признаков.

Выбор модели обоснован следующими причинами. Во-первых, ансамбль способен учитывать сложные комбинации факторов, которые линейная модель описывает ограниченно. Во-вторых, усреднение по множеству деревьев повышает устойчивость к шуму в данных. В-третьих, контроль глубины деревьев и минимальных размеров узлов уменьшает риск переобучения по сравнению с одиночным деревом. Следовательно, переход к `RandomForestClassifier` является методическим усилением baseline, а не формальной заменой алгоритма.


## 7. Подбор гиперпараметров

Подбор выполняется через `RandomizedSearchCV` с трёхкратной кросс-валидацией и `scoring='f1'`. Кросс-валидация используется для устойчивой оценки обобщающей способности модели на нескольких разбиениях train-данных и позволяет не делать выводы по одному случайному сплиту.

`RandomizedSearchCV` выбран как вычислительно оправданный способ поиска в многомерном пространстве гиперпараметров. В отличие от полного перебора, он позволяет исследовать большее число разумных комбинаций при приемлемой стоимости вычислений, что особенно важно для ансамблевой модели.

Оптимизируемые параметры имеют прямой содержательный смысл: `n_estimators` отвечает за стабильность ансамбля, `max_depth` контролирует сложность деревьев и риск переобучения, `min_samples_split` и `min_samples_leaf` задают уровень регуляризации структуры дерева, а `max_features` влияет на разнообразие деревьев и итоговую устойчивость модели. Таким образом, поиск настроен не формально, а вокруг параметров, определяющих bias/variance баланс.


In [ ]:
rf_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

rf_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", rf_numeric_transformer, numeric_features),
        ("cat", rf_categorical_transformer, categorical_features),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

param_distributions = {
    "model__n_estimators": [200, 300, 500, 700, 900],
    "model__max_depth": [None, 10, 20, 30, 40, 50],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.8],
}

search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=0,
)

search.fit(X_train, y_train)

cv_results = pd.DataFrame(search.cv_results_).sort_values(
    by=["rank_test_score", "mean_test_score"],
    ascending=[True, False],
)
cv_summary = cv_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_samples_split",
        "param_model__min_samples_leaf",
        "param_model__max_features",
    ]
].head(5)
cv_summary["mean_test_score"] = cv_summary["mean_test_score"].round(4)
cv_summary["std_test_score"] = cv_summary["std_test_score"].round(4)

print("Best params:")
print(search.best_params_)
print(f"Best CV f1: {search.best_score_:.4f}")
print("Top CV configurations:")
display(cv_summary.reset_index(drop=True))


Best params:
{'model__n_estimators': 300, 'model__min_samples_split': 20, 'model__min_samples_leaf': 2, 'model__max_features': 0.8, 'model__max_depth': 40}
Best CV f1: 0.6833
Top CV configurations:


,rank_test_score,mean_test_score,std_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf,param_model__max_features
0,1,0.6833,0.0067,300,40,20,2,0.8
1,2,0.6792,0.0093,200,50,2,2,0.8
2,3,0.6783,0.0077,200,30,10,8,0.5
3,4,0.6781,0.0095,300,50,2,2,sqrt
4,5,0.6757,0.0086,900,None,20,1,log2


## 8. Оценка качества и сравнение с baseline

Основная метрика — `F1-score` для класса `>50K`. В данной задаче `F1` предпочтительнее accuracy, поскольку выборка несбалансирована, и высокая accuracy могла бы быть достигнута за счёт доминирующего класса `<=50K` без реального качества на положительном классе.

Интерпретация `F1` в этой постановке такова: чем выше значение, тем лучше модель одновременно контролирует precision и recall для класса `>50K`. Следовательно, рост `F1` относительно baseline означает именно улучшение качества выявления положительного класса, а не формальный прирост доли верных ответов.


In [ ]:
best_model = search.best_estimator_

y_test_pred = best_model.predict(X_test)
rf_test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

comparison_table = pd.DataFrame(
    {
        "model": [
            "Dummy",
            "LogisticRegression (ДЗ_2 baseline)",
            "RandomForest (tuned)",
        ],
        "f1_test": [dummy_f1, baseline_logreg_f1, rf_test_f1],
    }
)
comparison_table["f1_test"] = comparison_table["f1_test"].round(4)

rf_vs_baseline_gain = rf_test_f1 - baseline_logreg_f1

display(comparison_table)
print(f"ΔF1(RandomForest - LogisticRegression baseline): {rf_vs_baseline_gain:.4f}")


,model,f1_test
0,Dummy,0.0000
1,LogisticRegression (ДЗ_2 baseline),0.6558
2,RandomForest (tuned),0.6875


ΔF1(RandomForest - LogisticRegression baseline): 0.0317


## 8.1 Интерпретация качества

По результатам текущего запуска лучшая конфигурация показывает `F1 = 0.6833` на кросс-валидации и `F1 = 0.6875` на тестовой выборке. Для сравнения, baseline-модель `LogisticRegression` достигает `F1 = 0.6558`, поэтому абсолютный прирост составляет `+0.0317`.

Это означает, что ансамблевая модель заметно лучше выделяет класс `>50K`, сохраняя более удачный баланс между precision и recall. Близость CV- и test-оценок указывает на приемлемую устойчивость результата и отсутствие выраженного переобучения.


## 9. Глобальная интерпретация: permutation importance

Permutation importance измеряет падение `F1` при случайном перемешивании отдельного признака. Если после такой перестановки качество снижается заметно, признак действительно используется моделью для принятия решений.

Метод важен тем, что даёт модельно-независимую глобальную оценку влияния признаков именно на отложенной выборке, а не только на данных обучения.


In [ ]:
perm_result = permutation_importance(
    estimator=best_model,
    X=X_test,
    y=y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring="f1",
)

perm_df = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    }
).sort_values("importance_mean", ascending=False)

n_display = min(20, len(perm_df))
perm_top = perm_df.head(n_display).sort_values("importance_mean", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(
    perm_top["feature"],
    perm_top["importance_mean"],
    xerr=perm_top["importance_std"],
    color="#2563EB",
)
plt.xlabel("Mean decrease in F1 after permutation")
plt.ylabel("Feature")
plt.title("Permutation Importance on the Test Set")
plt.tight_layout()

perm_fig_path = FIGURES_DIR / "permutation_importance_top20.png"
plt.savefig(perm_fig_path, dpi=220, bbox_inches="tight")
plt.show()

display(perm_df.head(10))


,feature,importance_mean,importance_std
5,marital_status,0.146733,0.005656
10,capital_gain,0.094537,0.003089
4,education_num,0.072006,0.007624
0,age,0.058434,0.006242
6,occupation,0.042869,0.003753
11,capital_loss,0.031821,0.002827
12,hours_per_week,0.027244,0.002941
7,relationship,0.008340,0.002034
1,workclass,0.006337,0.002314
8,race,0.001388,0.001612


## 10. Глобальная интерпретация: SHAP summary

SHAP summary дополняет permutation-анализ: он показывает не только относительную значимость признаков, но и направление их вклада в вероятность положительного класса.

Это особенно важно для ансамблевой модели, где отсутствуют простые линейные коэффициенты. SHAP позволяет интерпретировать решение модели в том признаковом пространстве, которое реально формируется после `ColumnTransformer`.


In [ ]:
def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


def extract_positive_class_shap_values(shap_values):
    if hasattr(shap_values, "values"):
        shap_values = shap_values.values
    if isinstance(shap_values, list):
        return shap_values[1]
    if getattr(shap_values, "ndim", 0) == 3:
        return shap_values[:, :, 1]
    return shap_values


preprocessor = best_model.named_steps["preprocessor"]
rf_estimator = best_model.named_steps["model"]

sample_size = min(1500, len(X_test))
X_test_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)
X_test_sample_transformed = to_dense(preprocessor.transform(X_test_sample))
transformed_feature_names = preprocessor.get_feature_names_out()

shap_explainer = shap.TreeExplainer(rf_estimator)
shap_values_raw = shap_explainer.shap_values(X_test_sample_transformed)
shap_values_positive = extract_positive_class_shap_values(shap_values_raw)

shap.summary_plot(
    shap_values_positive,
    X_test_sample_transformed,
    feature_names=transformed_feature_names,
    max_display=20,
    show=False,
)
plt.title("SHAP Summary Plot for Class >50K")
plt.tight_layout()

shap_fig_path = FIGURES_DIR / "shap_summary_top20.png"
plt.savefig(shap_fig_path, dpi=220, bbox_inches="tight")
plt.show()


## 10.1 Интерпретация глобальных результатов

Глобальные методы выделяют согласованный набор ключевых признаков: `marital_status`, `capital_gain`, `education_num`, `age`, `occupation`, `capital_loss`, `hours_per_week`. Такая структура выглядит логично: вероятность дохода `>50K` определяется накопленным капиталом, уровнем квалификации, жизненным этапом и характером занятости.

Неожиданного доминирования второстепенных или трудно объяснимых признаков не наблюдается. Напротив, модель опирается преимущественно на экономически и социально интерпретируемые факторы, что повышает доверие к содержательности результата.


## 11. Локальная интерпретация: выбор объектов

Для локального анализа выбираются три наблюдения тестовой выборки: объект с максимальной вероятностью `>50K`, пограничный объект с вероятностью около `0.5` и объект с минимальной вероятностью `>50K`. Такое построение позволяет проверить поведение модели на уверенных и неуверенных решениях.

Подобный выбор методически оправдан: он показывает не только крайние случаи, но и поведение модели вблизи порога классификации, где интерпретация особенно важна.


In [ ]:
test_proba = best_model.predict_proba(X_test)[:, 1]

idx_high = int(np.argmax(test_proba))
idx_low = int(np.argmin(test_proba))
idx_border = int(np.argmin(np.abs(test_proba - 0.5)))

selected_positions = []
for idx in [idx_high, idx_border, idx_low]:
    if idx not in selected_positions:
        selected_positions.append(idx)

selected_positions = selected_positions[:3]
selected_X = X_test.iloc[selected_positions]
selected_y_true = y_test.iloc[selected_positions]
selected_proba = best_model.predict_proba(selected_X)[:, 1]
selected_pred = (selected_proba >= 0.5).astype(int)

label_map = {0: "<=50K", 1: ">50K"}

local_cases = pd.DataFrame(
    {
        "test_row_index": selected_X.index,
        "true_label": [label_map[val] for val in selected_y_true.values],
        "pred_label": [label_map[val] for val in selected_pred],
        "proba_>50K": np.round(selected_proba, 4),
    }
).reset_index(drop=True)

case_file_labels = [f"case_{i + 1}" for i in range(len(local_cases))]
local_cases["case_id"] = case_file_labels

display(local_cases)


,test_row_index,true_label,pred_label,proba_>50K,case_id
0,29755,>50K,>50K,1.0000,case_1
1,39438,>50K,>50K,0.5004,case_2
2,518,<=50K,<=50K,0.0000,case_3


## 12. Локальная интерпретация: SHAP

Локальные объяснения строятся с помощью SHAP. Для каждого выбранного наблюдения выделяются признаки, которые увеличивают и уменьшают вероятность класса `>50K`, что позволяет перейти от усреднённой глобальной картины к анализу конкретных индивидуальных решений.

Такой подход нужен для проверки того, что модель принимает решения по содержательным причинам, а не только демонстрирует приемлемую итоговую метрику.


In [ ]:
X_selected_transformed = to_dense(preprocessor.transform(selected_X))
local_explainer = shap.TreeExplainer(rf_estimator)
local_shap_raw = local_explainer.shap_values(X_selected_transformed)
local_shap_positive = extract_positive_class_shap_values(local_shap_raw)

local_explanations = []
local_commentary = []

for i in range(len(selected_positions)):
    contrib = local_shap_positive[i]
    top_idx = np.argsort(np.abs(contrib))[-10:][::-1]

    exp_df = pd.DataFrame(
        {
            "feature": transformed_feature_names[top_idx],
            "contribution_to_>50K": contrib[top_idx],
        }
    )

    top_positive = exp_df[exp_df["contribution_to_>50K"] > 0]["feature"].head(3).tolist()
    top_negative = exp_df[exp_df["contribution_to_>50K"] < 0]["feature"].head(3).tolist()

    local_explanations.append(
        {
            "case_id": case_file_labels[i],
            "method": "SHAP",
            "top_positive_features": ", ".join(top_positive) if top_positive else "-",
            "top_negative_features": ", ".join(top_negative) if top_negative else "-",
        }
    )

    if selected_pred[i] == 1:
        explanation = (
            f"{case_file_labels[i]}: решение в пользу >50K поддержано признаками "
            f"{', '.join(top_positive[:2]) if top_positive else '-'}, тогда как вниз прогноз тянут "
            f"{', '.join(top_negative[:2]) if top_negative else '-'}"
        )
    else:
        explanation = (
            f"{case_file_labels[i]}: решение против >50K в основном определяется признаками "
            f"{', '.join(top_negative[:2]) if top_negative else '-'}, при этом вверх вероятность двигают "
            f"{', '.join(top_positive[:2]) if top_positive else '-'}"
        )
    local_commentary.append(explanation)

    colors = np.where(exp_df["contribution_to_>50K"] >= 0, "#16A34A", "#DC2626")
    plt.figure(figsize=(10, 5))
    plt.barh(exp_df["feature"], exp_df["contribution_to_>50K"], color=colors)
    plt.axvline(0, color="#374151", linewidth=1, linestyle="--")
    plt.gca().invert_yaxis()
    plt.xlabel("Contribution to probability of class >50K")
    plt.ylabel("Feature")
    plt.title(f"Local SHAP explanation: {case_file_labels[i]} (p={selected_proba[i]:.4f})")
    plt.tight_layout()

    local_fig_path = FIGURES_DIR / f"local_explanation_case_{i + 1}.png"
    plt.savefig(local_fig_path, dpi=220, bbox_inches="tight")
    plt.show()

    display(exp_df)

local_explanations_df = pd.DataFrame(local_explanations)
local_commentary_df = pd.DataFrame({"interpretation": local_commentary})

display(local_explanations_df)
display(local_commentary_df)


,feature,contribution_to_>50K
0,num__capital_gain,0.526262
1,cat__marital_status_Married-civ-spouse,0.189152
2,cat__workclass_Federal-gov,0.027364
3,cat__relationship_Husband,0.024214
4,num__age,0.014774
5,num__education_num,-0.012770
6,cat__occupation_Exec-managerial,-0.006615
7,num__capital_loss,-0.005319
8,num__fnlwgt,-0.004957
9,cat__occupation_Adm-clerical,0.004667


,feature,contribution_to_>50K
0,cat__marital_status_Married-civ-spouse,0.179958
1,num__education_num,0.096014
2,num__age,-0.063479
3,num__hours_per_week,0.040555
4,num__capital_gain,-0.038777
5,cat__relationship_Husband,0.037621
6,cat__occupation_Tech-support,0.026922
7,num__capital_loss,-0.020730
8,cat__workclass_Self-emp-inc,0.020604
9,cat__occupation_Exec-managerial,-0.018012


,feature,contribution_to_>50K
0,cat__marital_status_Married-civ-spouse,-0.088566
1,num__age,-0.080177
2,num__capital_gain,-0.021866
3,num__education_num,-0.018005
4,cat__relationship_Husband,-0.015535
5,num__hours_per_week,-0.013893
6,cat__occupation_Prof-specialty,0.010203
7,num__capital_loss,-0.005921
8,cat__occupation_Exec-managerial,-0.003569
9,cat__education_HS-grad,0.002946


,case_id,method,top_positive_features,top_negative_features
0,case_1,SHAP,"num__capital_gain, cat__marital_status_Married...","num__education_num, cat__occupation_Exec-manag..."
1,case_2,SHAP,"cat__marital_status_Married-civ-spouse, num__e...","num__age, num__capital_gain, num__capital_loss"
2,case_3,SHAP,"cat__occupation_Prof-specialty, cat__education...","cat__marital_status_Married-civ-spouse, num__a..."


,interpretation
0,case_1: решение в пользу >50K поддержано призн...
1,case_2: решение в пользу >50K поддержано призн...
2,case_3: решение против >50K в основном определ...


## 12.1 Интерпретация локальных результатов

Для `case_1` вероятность класса `>50K` равна `1.0000`. Решение в пользу высокого дохода почти полностью поддерживается вкладом `capital_gain` и индикатором `marital_status_Married-civ-spouse`, что соответствует ожидаемой логике уверенного положительного прогноза.

`case_2` является пограничным (`0.5004`). Вероятность вверх двигают признаки, связанные с `marital_status_Married-civ-spouse`, `education_num` и `hours_per_week`, тогда как вниз её тянут `age`, `capital_gain` и `capital_loss`. Конкуренция факторов противоположного знака объясняет близость решения к порогу классификации.

Для `case_3` вероятность класса `>50K` равна `0.0000`. Отрицательный прогноз определяется вкладом признаков `marital_status_Married-civ-spouse`, `age`, `capital_gain` и `education_num`. Такая структура характерна для уверенного решения против положительного класса и выглядит содержательно обоснованной.


## 13. Мое мнение

С точки зрения предметной области модель демонстрирует адекватное поведение. В числе главных факторов доминируют капитал, семейный статус, образование, возраст и занятость, то есть признаки, которые действительно связаны с различиями в доходе. Это подтверждает, что модель использует содержательные закономерности, а не случайные корреляции.

Одновременно модель не свободна от риска систематического смещения. Несмотря на низкий глобальный вклад отдельных демографических признаков, в данных присутствуют переменные и прокси-факторы, которые потенциально способны влиять на индивидуальные решения. Следовательно, модель подходит для учебного анализа и сравнительного исследования качества, но для прикладного использования требует отдельной fairness-проверки.


## 14. Итоговый вывод

1. В качестве основной модели используется `RandomForestClassifier` в составе `Pipeline` с `ColumnTransformer`.
2. Лучшая конфигурация по результатам поиска: `n_estimators=300`, `max_depth=40`, `min_samples_split=20`, `min_samples_leaf=2`, `max_features=0.8`.
3. Качество модели составило `F1 = 0.6833` на кросс-валидации и `F1 = 0.6875` на тестовой выборке.
4. Baseline `LogisticRegression` показал `F1 = 0.6558`, поэтому прирост ансамбля равен `+0.0317`.
5. Наиболее важными глобальными факторами оказались `marital_status`, `capital_gain`, `education_num`, `age`, `occupation`, `capital_loss`, `hours_per_week`.
6. Глобальная и локальная интерпретация показали, что модель опирается на экономически и социально содержательные признаки; уверенные положительные решения поддерживаются капиталом и устойчивым семейно-трудовым статусом, а отрицательные — отсутствием этих сигналов.
7. Итог: модель даёт интерпретируемое и статистически обоснованное улучшение относительно baseline и может считаться состоятельной для данной учебной постановки.
